💡 **Environment:** `clamp-analyses`

# ARCHS4 CRISPR-Cas9 - Biology of significant LVs

For every CRISPR-significant LV (FDR < 0.05, selected in `00_gene_enrichment_CRISPRCas9.ipynb`), builds three tables, one row per LV: significant ORA gene sets (GO:BP, canonical pathways, CellMarker, ARCHS4 cell-lines, Azimuth), significant trait associations (canonical-model GLS, PhenomeXcan phenotypes), and the top 1% LINCS L1000 perturbagens by projection score.


## Load inputs

In [ ]:
library(dplyr)
library(readr)
library(tidyr)
library(knitr)
library(IRdisplay)

fdr_threshold <- snakemake@params[["fdr"]]

lv_sig <- readr::read_csv(snakemake@input[["lv_sig"]], show_col_types = FALSE)$lv
lv_order_numeric <- lv_sig[order(as.integer(sub("^LV", "", lv_sig)))]
cat("Significant LVs:", length(lv_sig), "\n")

show_full <- function(df) {
  md <- paste(as.character(knitr::kable(as.data.frame(df), format = "markdown")), collapse = "\n")
  IRdisplay::display_markdown(md)
}


## ORA table

In [ ]:
read_ora <- function(ora_dir) {
  readr::read_csv(file.path(ora_dir, "enrichment.csv.gz"), show_col_types = FALSE) %>%
    dplyr::filter(LV %in% lv_sig, p.adjust < fdr_threshold)
}

bp_gmt_lines <- readr::read_lines(snakemake@input[["bp_gmt"]])
bp_names <- do.call(rbind, lapply(bp_gmt_lines, function(line) {
  parts <- strsplit(line, "\t")[[1]]
  data.frame(ID = parts[1], name = parts[2], stringsAsFactors = FALSE)
}))

ora_sources <- list(
  bp                 = snakemake@input[["ora_bp"]],
  canonical          = snakemake@input[["ora_canonical"]],
  cellmarker         = snakemake@input[["ora_cellmarker"]],
  archs4_cell_lines  = snakemake@input[["ora_archs4_cell_lines"]],
  azimuth            = snakemake@input[["ora_azimuth"]]
)

ora_summary <- function(db_name, ora_dir) {
  df <- read_ora(ora_dir)
  if (db_name == "bp") {
    df <- df %>%
      dplyr::left_join(bp_names, by = "ID") %>%
      dplyr::mutate(label = dplyr::coalesce(name, ID))
  } else {
    df <- df %>% dplyr::mutate(label = ID)
  }
  df %>%
    dplyr::group_by(LV) %>%
    dplyr::summarise(!!db_name := paste(sort(unique(label)), collapse = "; "), .groups = "drop")
}

ora_table <- tibble::tibble(LV = lv_order_numeric)
for (db_name in names(ora_sources)) {
  ora_table <- ora_table %>%
    dplyr::left_join(ora_summary(db_name, ora_sources[[db_name]]), by = "LV")
}
ora_table <- ora_table %>%
  dplyr::mutate(dplyr::across(-LV, ~ tidyr::replace_na(.x, "")))

readr::write_csv(ora_table, snakemake@output[["ora_table"]])
cat("ORA table:", nrow(ora_table), "rows\n")
show_full(ora_table)


## Traits table

In [ ]:
canonical_gls <- readr::read_tsv(snakemake@input[["canonical_gls"]], show_col_types = FALSE)

traits_table <- canonical_gls %>%
  dplyr::filter(lv %in% lv_sig, fdr < fdr_threshold) %>%
  dplyr::group_by(lv) %>%
  dplyr::summarise(
    traits = paste(sort(unique(phenotype_desc)), collapse = "; "),
    .groups = "drop"
  ) %>%
  dplyr::rename(LV = lv) %>%
  dplyr::right_join(tibble::tibble(LV = lv_order_numeric), by = "LV") %>%
  dplyr::mutate(traits = tidyr::replace_na(traits, "")) %>%
  dplyr::arrange(match(LV, lv_order_numeric))

readr::write_csv(traits_table, snakemake@output[["traits_table"]])
cat("Traits table:", nrow(traits_table), "rows\n")
show_full(traits_table)


## LINCS table

In [ ]:
lincs_top <- readr::read_csv(snakemake@input[["lincs_top"]], show_col_types = FALSE)

drug_names <- readr::read_tsv(snakemake@input[["drug_names"]], show_col_types = FALSE) %>%
  dplyr::select(drugbank_id, drugbank_name) %>%
  dplyr::distinct(drugbank_id, .keep_all = TRUE)

lincs_table <- lincs_top %>%
  dplyr::inner_join(drug_names, by = c("perturbagen" = "drugbank_id")) %>%
  dplyr::group_by(lv) %>%
  dplyr::summarise(
    lincs_top_drugs = paste(sort(unique(drugbank_name)), collapse = "; "),
    .groups = "drop"
  ) %>%
  dplyr::rename(LV = lv) %>%
  dplyr::right_join(tibble::tibble(LV = lv_order_numeric), by = "LV") %>%
  dplyr::mutate(lincs_top_drugs = tidyr::replace_na(lincs_top_drugs, "")) %>%
  dplyr::arrange(match(LV, lv_order_numeric))

readr::write_csv(lincs_table, snakemake@output[["lincs_table"]])
cat("LINCS table:", nrow(lincs_table), "rows\n")
show_full(lincs_table)
